# ** Feature Engineering & Transformation**  
 **Notebook:** `02_feature_engineering.ipynb`  

## ** Objective**  
The goal of this notebook is to **enhance our dataset** by engineering new features, scaling numerical variables, and reducing dimensionality to improve model performance.  



 **Let'sss startt!** 


### Import libraries 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn


### Load the dataset

In [ ]:
file_path = "../data/processed_data.csv"  
df_cleaned = pd.read_csv(file_path)


df_cleaned.head()


### ? **1. Scaling & Normalization**  

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler


standard_scaler = StandardScaler()


df_cleaned['AdSpend_Standard'] = standard_scaler.fit_transform(df_cleaned[['AdSpend']])
df_cleaned['TimeOnSite_Standard'] = standard_scaler.fit_transform(df_cleaned[['TimeOnSite']])

df_cleaned.drop(columns=['AdSpend', 'TimeOnSite'], inplace=True)


StandardScaler standardizes data with mean = 0 and std = 1

MinMaxScaler forces all values into [0,1] range, which may compress important variations in high-value features like AdSpend.

### ? **2. Feature Creation**  

In [ ]:

df_cleaned['Engagement_Score'] = (
    df_cleaned['WebsiteVisits'] * df_cleaned['PagesPerVisit'] * df_cleaned['TimeOnSite_Standard']
)


df_cleaned['Marketing_Efficiency'] = df_cleaned['ConversionRate'] / (df_cleaned['AdSpend_Standard'] + 1e-5) 

Engagement Score quantifies user activity (more visits, pages, and time ? higher score).

Marketing Spend Efficiency helps measure how well ad spend translates into conversions.

In [ ]:
df_cleaned

### ? **3. Dimensionality Reduction (Feature Selection)** 

In [ ]:
from sklearn.feature_selection import mutual_info_classif


features = df_cleaned.drop(columns=['Conversion'])
target = df_cleaned['Conversion']


mi_scores = mutual_info_classif(features, target, random_state=42)
feature_importance = pd.Series(mi_scores, index=features.columns).sort_values(ascending=False)

top_features = feature_importance[feature_importance > 0.01].index.tolist()
df_selected = df_cleaned[top_features + ['Conversion']] 


Mutual Information (MI) measures feature relevance to Conversion.

We keep only important features, improving model efficiency without losing key information.

In [ ]:

df_selected.to_csv("../data/feature_engineered.csv.csv", index=False)

print("? feature data saved successfully as 'feature_engineered' in the 'data' folder.")
